In [1]:
# Imports
import os
import torch
import cv2
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.models import resnet50
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize
from sklearn.neighbors import NearestNeighbors

In [2]:
# # First color masking is applied to use only the frames with a certain amount of green (to remove the floor from videos, preventing formation of a false cluster)
# # Variables
# video_path = "data/input_videos/grasrobot.mp4"

# # Open input video
# video = cv2.VideoCapture(video_path)

# green_samples = []

# # Sample few samples evenly from video
# frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
# sample_frames = 100
# step = max(1, frame_count // sample_frames) # if sample_frames to big, step size = 1

# for i in range(0, frame_count, step):
#     video.set(cv2.CAP_PROP_POS_FRAMES, i)
#     ret, frame = video.read()
#     if not ret:
#         continue

#     # Convert to HSV
#     hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

#     # Extract greenlike pixels with broad filter
#     lower = np.array([35, 35, 35])
#     upper = np.array([90, 255, 255])
#     mask = cv2.inRange(hsv, lower, upper)

#     green_pixels = hsv[mask > 0]
#     green_samples.append(green_pixels)


# # Stack all green and determine min/max
# if green_samples:
#     green = np.vstack(green_samples)

#     lower_green = np.min(green, axis=0)
#     upper_green = np.max(green, axis=0)

#     print(f"Green range: {lower_green} to {upper_green}")
# else:
#     print("No green pixels detected")

# # Close input video
# video.release()

# # Visualize distribution of green pixels on HUE value
# hues = green[:, 0]
# hist = np.histogram(hues, bins=180, range=(0, 179))[0]

# plt.plot(hist)
# plt.title("Hue Distribution")
# plt.xlabel("Hue value")
# plt.ylabel("Pixel count")
# plt.show()

# # Add padding of 5% to remove outliers (assumingly non-grass objects)
# lower_green = np.percentile(green, 5, axis=0)
# upper_green = np.percentile(green, 95, axis=0)

# print(f"Refined green range (percentile-based):")
# print("Lower:", lower_green.astype(int))
# print("Upper:", upper_green.astype(int))

In [3]:
# output_path = "data/input_videos/grasbot_green.mp4"

# # Open input video
# video = cv2.VideoCapture(video_path)

# # Video properties
# fps = int(video.get(cv2.CAP_PROP_FPS))
# width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
# height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
# fourcc = cv2.VideoWriter_fourcc('m', 'p', '4', 'v')
# out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

# # Threshold: minimum % of green pixels to keep frame
# green_threshold = 0.75

# while True:
#     read, frame = video.read()
#     if not read:
#         break

#     # Convert to hsv
#     hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

#     # Create green area mask
#     mask = cv2.inRange(hsv, lower_green, upper_green)

#     # Calc % of green pixels
#     green_ratio = np.sum(mask > 0) / (frame.shape[0] * frame.shape[1])

#     # Keep frame if threshold reached
#     if green_ratio >= green_threshold:
#         out.write(frame)

# video.release()
# out.release()

In [4]:
# # Load and save dataset from video
# # Variables
# video_path = "data/input_videos/grasrobot_green.mp4"
# image_path = "data/grass1/"
# amount_of_frames = 100

# # Extract frames
# video = cv2.VideoCapture(video_path)
# frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
# frame_interval = int(frame_count/(amount_of_frames-1)) # -1 since frames start at 0
# current_frame = 0

# # Loop through every frame
# while True:
#     read, frame = video.read()
#     if not read:
#         break # if no frame read, end of video
#     if current_frame % frame_interval == 0:
#         img_name = os.path.join(image_path, f"frame_{current_frame:04d}.jpg")
#         cv2.imwrite(img_name, frame)
#         print(f"Saved frame {current_frame}")
#     current_frame += 1

# # Release video capture object
# video.release()
# print(f"All frames have been saved to {image_path}")

In [5]:
# # Load and save dataset from video
# # Variables
# video_path = "data/input_videos/grasrobot.mp4"
# image_path = "data/grass/"
# amount_of_frames = 100

# # Extract frames
# video = cv2.VideoCapture(video_path)
# frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
# frame_interval = int(frame_count/(amount_of_frames-1)) # -1 since frames start at 0
# current_frame = 0

# # Loop through every frame
# while True:
#     read, frame = video.read()
#     if not read:
#         break # if no frame read, end of video
#     if current_frame % frame_interval == 0:
#         img_name = os.path.join(image_path, f"frame_{current_frame:04d}.jpg")
#         cv2.imwrite(img_name, frame)
#         print(f"Saved frame {current_frame}")
#     current_frame += 1

# # Release video capture object
# video.release()
# print(f"All frames have been saved to {image_path}")

In [6]:
# Load functions
def preprocess_image(image):
        preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
                             ])
        return preprocess(image)

In [7]:
num_classes = 1081
resnet_checkpoint_path = 'models/pretrained/resnet50_weights_best_acc.tar'

try:
    model = resnet50(num_classes=num_classes)
    model = torch.nn.Sequential(*(list(model.children())[:-1]))
    checkpoint = torch.load(resnet_checkpoint_path, map_location='cpu', weights_only=True)
    state_dict = checkpoint.get('state_dict', checkpoint)
    model.load_state_dict(state_dict=state_dict, strict=False)
    print("ResNet loaded succesfully")
except Exception as e:
    print(f"Error loading ResNet: {e}")

model.eval()

ResNet loaded succesfully


Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)


In [8]:
frames = os.listdir("data/grass/")
embeddings = []

for _, frame in enumerate(frames):
    img = Image.open("data/grass/" + frame)
    preprocessed_img = preprocess_image(img)
    input_tensor = preprocessed_img.unsqueeze(0)
    with torch.no_grad():
        features = model(input_tensor)
    embeddings.append(features.flatten())
    
embeddingspace = np.array(embeddings)
embeddingspace_norm = normalize(embeddingspace, axis=1)

In [9]:
# kNN density estimation
# Dimensionality reduction
pca = PCA(n_components=50, random_state=42)
embeddingspace_reduced = pca.fit_transform(embeddingspace)

k = 20
nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='auto', metric='cosine').fit(embeddingspace_reduced)
distances, indices = nbrs.kneighbors(embeddingspace_reduced)

# Remove distance to self (first column)
distances = distances[:, 1:]

density = 1 / (np.mean(distances, axis=1) + 1e-10)
average_density = np.mean(density)

above_average = []
for d in density:
    if d > average_density:
        above_average.append(d)

threshold = np.mean(above_average)

print(average_density)
print(threshold)

19.198776
36.01091


In [10]:
frames = os.listdir("data/grass1/")

json_file_path = "data/json_files/embedding.json"
json_data = []

embeddings = []

for _, frame in enumerate(frames):
    img = Image.open("data/grass1/" + frame)
    preprocessed_img = preprocess_image(img)
    input_tensor = preprocessed_img.unsqueeze(0)
    with torch.no_grad():
        features = model(input_tensor)

    data = features.cpu().numpy().flatten().tolist()
    json_data.append(data)

    embeddings.append(features.cpu().numpy().flatten())

with open(json_file_path, 'w') as json_file:
    json.dump(json_data, json_file, indent=4)
    
embedding1 = np.array(embeddings)
embedding1_norm = normalize(embedding1, axis=1)

In [11]:
# kNN density estimation
# Dimensionality reduction
pca = PCA(n_components=50, random_state=42)
embeddingspace_reduced = pca.fit_transform(embedding1)

k = 20
nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='auto', metric='cosine').fit(embeddingspace_reduced)
distances, indices = nbrs.kneighbors(embeddingspace_reduced)

# Remove distance to self (first column)
distances = distances[:, 1:]

density = 1 / (np.mean(distances, axis=1) + 1e-10)
average_density = np.mean(density)

print(average_density)

18.714827


In [12]:
# Variables
image_folder = "data/grass"
embedding_json = "data/json_files/embedding.json"
k = 20
embedding_space_max_size = 100

# Functions
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def preprocess_image(img_path):
    image = Image.open(img_path).convert("RGB")
    return transform(image).unsqueeze(0)  # Add batch dim

def get_embedding(model, image_tensor):
    with torch.no_grad():
        features = model(image_tensor)
    return features.squeeze().numpy()

def compute_density(new_emb, emb_space, k):
    if len(emb_space) < k:
        return float('inf')  # Treat as low density
    nbrs = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
    nbrs.fit(emb_space)
    distances, _ = nbrs.kneighbors([new_emb])
    return np.mean(distances)

def load_embedding_space(json_path):
    if os.path.exists(json_path):
        with open(json_path, 'r') as f:
            data = json.load(f)
        if isinstance(data, dict) and "embeddings" in data:
            return data["embeddings"]
        elif isinstance(data, list):
            return data  # it's already a list of embeddings
    return []

def save_embedding_space(json_path, embedding_list):
    with open(json_path, 'w') as f:
        json.dump({"embeddings": embedding_list}, f)

# Main process
def process_folder(folder_path, model, json_path, k, threshold, max_size):
    image_files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    emb_space = load_embedding_space(json_path)

    for img_name in image_files:
        img_path = os.path.join(folder_path, img_name)
        img_tensor = preprocess_image(img_path)
        emb = get_embedding(model, img_tensor)

        emb_space_np = np.array(emb_space) if emb_space else np.empty((0, len(emb)))
        density = compute_density(emb, emb_space_np, k)

        if density < threshold:
            print(f"{img_name}: mow")
        else:
            print(f"{img_name}: don't mow")
            emb_space.append(emb.tolist())
            if len(emb_space) > max_size:
                emb_space = emb_space[-max_size:]
            save_embedding_space(json_path, emb_space)

# Run code
process_folder(
    image_folder,
    model,
    embedding_json,
    k=k,
    threshold=threshold,
    max_size=embedding_space_max_size
)

frame_0000.jpg: mow
frame_0084.jpg: don't mow
frame_0168.jpg: don't mow
frame_0252.jpg: don't mow
frame_0336.jpg: mow
frame_0420.jpg: mow
frame_0504.jpg: mow
frame_0588.jpg: mow
frame_0672.jpg: mow
frame_0756.jpg: mow
frame_0840.jpg: don't mow
frame_0924.jpg: don't mow
frame_1008.jpg: mow
frame_1092.jpg: mow
frame_1176.jpg: don't mow
frame_1260.jpg: mow
frame_1344.jpg: mow
frame_1428.jpg: don't mow
frame_1512.jpg: mow
frame_1596.jpg: mow
frame_1680.jpg: don't mow
frame_1764.jpg: mow
frame_1848.jpg: mow
frame_1932.jpg: mow
frame_2016.jpg: don't mow
frame_2100.jpg: don't mow
frame_2184.jpg: mow
frame_2268.jpg: mow
frame_2352.jpg: mow
frame_2436.jpg: don't mow
frame_2520.jpg: don't mow
frame_2604.jpg: don't mow
frame_2688.jpg: don't mow
frame_2772.jpg: don't mow
frame_2856.jpg: don't mow
frame_2940.jpg: don't mow
frame_3024.jpg: mow
frame_3108.jpg: mow
frame_3192.jpg: don't mow
frame_3276.jpg: mow
frame_3360.jpg: don't mow
frame_3444.jpg: don't mow
frame_3528.jpg: mow
frame_3612.jpg: don'

In [13]:
embedding2_path = "data/json_files/embedding.json" 
embedding2 = load_embedding_space(embedding2_path)

embedding2 = np.array(embedding2)

print(embedding2.shape)

# kNN density estimation
# Dimensionality reduction
pca = PCA(n_components=50, random_state=42)
embeddingspace_reduced = pca.fit_transform(embedding1)

k = 20
nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='auto', metric='cosine').fit(embeddingspace_reduced)
distances, _ = nbrs.kneighbors(embeddingspace_reduced)

# Remove distance to self (first column)
distances = distances[:, 1:]

density = 1 / (np.mean(distances, axis=1) + 1e-10)
average_density = np.mean(density)

print(average_density)

(100, 2048)
18.714827
